# C4 scaling study — does the from-scratch vs pretrained gap open with scale?

**Runtime: L4 GPU + High-RAM.** Expect 2–3 h. The sweep cell resumes automatically if the
runtime disconnects — just re-run it.

**Public data only (AG News).** No MIMIC, no credentialed data anywhere in this notebook,
by design. The clinical scaling run stays future work requiring a cluster with credentialed
access.

Background: C37 found a 10.6M-parameter from-scratch encoder statistically indistinguishable
from PubMedBERT (109M, 3.1B words) on the clinical task — gap +0.0031, Welch p = 0.690. This
asks whether that gap opens at 30M and 100M.

### 1. Install

In [1]:
!pip -q install -U datasets transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 151.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 156.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.7 MB/s eta 0:00:00


### 2. Imports, GPU check, mount Drive

In [2]:
import os, json, time, math, statistics as st
import numpy as np, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score
from scipy import stats

assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > L4 GPU"
print(f"device: {torch.cuda.get_device_name(0)}   torch {torch.__version__}")

from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/thesis/results_2026-08/scaling'
os.makedirs(OUT, exist_ok=True)
RESULTS_PATH = f"{OUT}/scaling_results.json"
print("results ->", RESULTS_PATH)

device: NVIDIA A100-SXM4-80GB   torch 2.11.0+cu128
Mounted at /content/drive
results -> /content/drive/MyDrive/thesis/results_2026-08/scaling/scaling_results.json


### 3. Data

In [3]:
ds  = load_dataset("fancyzhx/ag_news")
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
MAXLEN, N_TRAIN = 256, 40_000
rng = np.random.default_rng(42)
tr_idx = rng.choice(len(ds['train']), N_TRAIN, replace=False)

def encode(split, idx=None):
    d = split.select(idx) if idx is not None else split
    e = tok(d['text'], truncation=True, padding='max_length',
            max_length=MAXLEN, return_tensors='pt')
    return e['input_ids'], e['attention_mask'], torch.tensor(d['label'])

Xtr, Mtr, ytr = encode(ds['train'], tr_idx)
Xte, Mte, yte = encode(ds['test'])
print(f"train {tuple(Xtr.shape)}  test {tuple(Xte.shape)}  vocab {tok.vocab_size}")

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/ag_news@eb185aade064a813bc0b7f42de02595523103ca4/.huggingface.yaml'. Repository id must be 'namespace/name', got 'ag_news'.

### 4. Model definitions — **check the parameter counts before continuing**

In [4]:
class Enc(nn.Module):
    """Pre-LN transformer encoder. Non-embedding params ~ 12*d^2*L."""
    def __init__(self, vocab, d, L, heads, n_cls=4, maxlen=MAXLEN):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(maxlen, d)
        layer = nn.TransformerEncoderLayer(d, heads, 4*d, 0.1, batch_first=True,
                                           norm_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, L)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_cls)
    def forward(self, ids, mask):
        p = torch.arange(ids.size(1), device=ids.device)
        h = self.enc(self.emb(ids) + self.pos(p), src_key_padding_mask=~mask.bool())
        h = (self.norm(h) * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
        return self.head(h)

def non_emb(m):
    """Non-embedding params. Handles our naming (emb./pos.) and HF BERT's
    (*.embeddings.*) so both arms are measured on the same basis."""
    e = sum(p.numel() for n, p in m.named_parameters()
            if 'embedding' in n.lower() or n.startswith(('emb.', 'pos.')))
    return sum(p.numel() for p in m.parameters()) - e

CONFIGS = {'10M': (384, 6, 6), '30M': (576, 8, 8), '100M': (768, 14, 12)}
print("parameter check — expect roughly 10.6M / 31.9M / 99.1M:")
for k, (d, L, h) in CONFIGS.items():
    print(f"  {k}: d={d} L={L} heads={h} -> {non_emb(Enc(tok.vocab_size, d, L, h)):,}")

NameError: name 'MAXLEN' is not defined

### 5. Training loop and resume state

In [ ]:
def run(model, seed, tag, epochs=3, bs=32, lr=3e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    dev = 'cuda'; model = model.to(dev)
    dl = DataLoader(TensorDataset(Xtr, Mtr, ytr), batch_size=bs, shuffle=True,
                    generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, epochs*len(dl))
    lossf = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler()
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        for i, (x, m, y) in enumerate(dl):
            x, m, y = x.to(dev), m.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                loss = lossf(model(x, m), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            if i % 300 == 0:
                print(f"  {tag} s{seed} ep{ep} {i}/{len(dl)} loss {loss.item():.3f}", flush=True)
    model.eval(); preds = []
    with torch.no_grad(), torch.cuda.amp.autocast():
        for i in range(0, len(Xte), 128):
            preds.append(model(Xte[i:i+128].to(dev), Mte[i:i+128].to(dev)).argmax(-1).cpu())
    p = torch.cat(preds)
    return {"tag": tag, "seed": seed, "non_emb_params": non_emb(model),
            "accuracy": float(accuracy_score(yte, p)),
            "macro_f1": float(f1_score(yte, p, average='macro')),
            "minutes": (time.time()-t0)/60}

class Wrap(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, ids, mask):
        return self.m(input_ids=ids, attention_mask=mask).logits

results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else []
done = {(r['tag'], r['seed']) for r in results}
print(f"{len(done)} runs already complete" if done else "starting fresh")

def save():
    json.dump(results, open(RESULTS_PATH, "w"), indent=2)

### 6. The sweep (long — resumable)

In [ ]:
for name, (d, L, h) in CONFIGS.items():
    for seed in [0, 1, 2]:
        tag = f"scratch_{name}"
        if (tag, seed) in done:
            continue
        r = run(Enc(tok.vocab_size, d, L, h), seed, tag)
        print(f"DONE {tag} s{seed}: acc {r['accuracy']:.4f} f1 {r['macro_f1']:.4f} ({r['minutes']:.1f}m)\n")
        results.append(r); save(); torch.cuda.empty_cache()

for seed in [0, 1, 2]:
    tag = "pretrained_bert"
    if (tag, seed) in done:
        continue
    m = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
    r = run(Wrap(m), seed, tag, lr=2e-5)
    print(f"DONE {tag} s{seed}: acc {r['accuracy']:.4f} ({r['minutes']:.1f}m)\n")
    results.append(r); save(); torch.cuda.empty_cache()

print("sweep complete")

### 7. Analysis — safe to re-run any time

In [ ]:
results = json.load(open(RESULTS_PATH))
by = {}
for r in results:
    by.setdefault(r['tag'], []).append(r['accuracy'])

print(f"{'arm':<20}{'non-emb params':>16}{'accuracy':>22}{'n':>4}")
for tag, a in by.items():
    p  = next(r['non_emb_params'] for r in results if r['tag'] == tag)
    sd = st.stdev(a) if len(a) > 1 else float('nan')
    print(f"{tag:<20}{p:>16,}{st.mean(a):>15.4f} ± {sd:.4f}{len(a):>4}")

pre = by.get('pretrained_bert', [])
print("\ngap vs pretrained (positive = pretrained ahead):")
for name in CONFIGS:
    s = by.get(f'scratch_{name}', [])
    if len(s) > 1 and len(pre) > 1:
        t, pv  = stats.ttest_ind(pre, s)
        pooled = math.sqrt((st.stdev(pre)**2 + st.stdev(s)**2) / 2)
        gap    = st.mean(pre) - st.mean(s)
        print(f"  {name:<6}{gap:+.4f}   {gap/pooled:>5.2f} pooled sd   Welch p={pv:.3f}")
    else:
        print(f"  {name:<6}(incomplete — {len(s)} scratch seeds, {len(pre)} pretrained)")

print(f"\nsaved -> {RESULTS_PATH}")